In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import joblib

# 1. Load datasets
X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv')

# 2. Align rows if there is a mismatch
min_rows = min(len(X_train), len(y_train))
X_train = X_train.iloc[:min_rows]
y_train = y_train.iloc[:min_rows]

# 3. Apply custom byte_ratio feature engineering
if 'src_bytes' in X_train.columns and 'dst_bytes' in X_train.columns:
    X_train['byte_ratio'] = X_train['src_bytes'] / (X_train['dst_bytes'] + 1)

# 4. FIX: Drop any non-numeric text columns (like column 33) so XGBoost doesn't crash
X_train = X_train.select_dtypes(include=[np.number])
X_train.fillna(0, inplace=True)

# 5. Encode target labels properly
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train.values.ravel())

# 6. Train and Save XGBoost Model
print("--- TRAINING MODEL #1: XGBOOST CLASSIFIER ---")
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train_encoded)

joblib.dump(xgb_model, 'xgboost_model.pkl')
print("✅ XGBoost Model trained and saved successfully as 'xgboost_model.pkl'!")

--- TRAINING MODEL #1: XGBOOST CLASSIFIER ---
✅ XGBoost Model trained and saved successfully as 'xgboost_model.pkl'!


In [6]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.preprocessing import StandardScaler

print("--- TRAINING MODEL #2: DEEP AUTOENCODER ---")

# 1. Scale features so the neural network trains smoothly
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

# 2. Build a simple Deep Autoencoder architecture
input_dim = X_scaled.shape[1]

input_layer = Input(shape=(input_dim,))
encoder = Dense(32, activation="relu")(input_layer)
encoder = Dense(16, activation="relu")(encoder)
decoder = Dense(32, activation="relu")(encoder)
output_layer = Dense(input_dim, activation="linear")(decoder)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# 3. Train the Autoencoder (learning normal patterns)
autoencoder.fit(X_scaled, X_scaled, epochs=10, batch_size=32, shuffle=True, verbose=1)

# 4. Save the trained Autoencoder model
autoencoder.save('autoencoder_model.keras')
print("✅ Autoencoder Model trained and saved successfully as 'autoencoder_model.keras'!")

--- TRAINING MODEL #2: DEEP AUTOENCODER ---
Epoch 1/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.7817
Epoch 2/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6341
Epoch 3/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5871
Epoch 4/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5595
Epoch 5/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5407
Epoch 6/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5274
Epoch 7/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5176
Epoch 8/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5110
Epoch 9/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5061
Epoch 10/10
350/350 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5019
✅ Autoencoder Model trained and saved successfully as 'autoencoder_model.keras'!


In [7]:
from sklearn.ensemble import IsolationForest
import joblib

print("--- TRAINING MODEL #3: ISOLATION FOREST ---")

# 1. Initialize and train the Isolation Forest
# contamination='auto' lets the model decide the outlier fraction automatically
iso_forest = IsolationForest(contamination='auto', random_state=42)
iso_forest.fit(X_train)

# 2. Save the trained Isolation Forest model
joblib.dump(iso_forest, 'isolation_forest_model.pkl')
print("✅ Isolation Forest Model trained and saved successfully as 'isolation_forest_model.pkl'!")

--- TRAINING MODEL #3: ISOLATION FOREST ---
✅ Isolation Forest Model trained and saved successfully as 'isolation_forest_model.pkl'!
